# 04 — Fine-tune Deep Learning Models for Sentiment + Toxicity

**In-domain rebuild (slang-aware).**

This notebook fine-tunes BERT / RoBERTa / DistilBERT for two tasks on Dota 2
in-game chat:

1. **Sentiment** (3-class: negative / neutral / positive) — trained **in-domain
   on the gold set** (`data/gold/train.csv`), *not* on external tweets.
2. **Toxicity** (multi-label, 6 labels) — trained on **Jigsaw** (external)
   because the gold set has too few positive toxic examples to supervise
   (`severe_toxic` and `threat` have **zero** positives).

### Why the previous results were *worse than chance*
The earlier pipeline fine-tuned sentiment on **TweetEval / SST-2** and only
evaluated on the Dota gold set. Tweet/movie polarity is *anti-correlated* with
Dota chat labelling — `gg` = positive sportsmanship, `g` = neutral engage call —
so the model produced a **negative MCC** (systematically wrong).

### Two fixes (this notebook)
- **Train in-domain** on the gold labels themselves.
- **Slang translation** (`src/preprocessing/slang.py`): rewrite terse jargon
  (`gg`, `g`, `ez`, `noob`, `kys`) into plain English **before** tokenization,
  so the pretrained embeddings actually fire on 1–2 token messages. The *same*
  translation runs at inference (notebook 05).

Imbalance handling: minority oversampling + weighted cross-entropy (sentiment);
`pos_weight` BCE (toxicity).


In [1]:
# Sel 0: dependencies (Colab/Kaggle)
%pip install -q datasets accelerate evaluate detoxify

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Sel 1: Setup — repo root, config, banner, run log
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('04_training', config)
run_log = RunLog(notebook='04_training', config_path='configs/experiment.yaml')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
else:
    msg = 'CUDA not available — training will be VERY slow on CPU. Use a GPU (Colab/Kaggle).'
    print(f'[WARN] {msg}')
    run_log.add_warning(msg)

SEED = int(config['seed'])

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 04_training
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: da158c0
Started at: 2026-06-10T01:42:31+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4
CUDA available: True
Device: NVIDIA GeForce RTX 3060


In [3]:
# Sel 2: Imports (reload so edits to src/ take effect without kernel restart)
import importlib
import src.preprocessing.slang
import src.training.gold_loaders
import src.training.sentiment_trainer
import src.training.toxicity_trainer
importlib.reload(src.preprocessing.slang)
importlib.reload(src.training.gold_loaders)
importlib.reload(src.training.sentiment_trainer)
importlib.reload(src.training.toxicity_trainer)

from src.preprocessing.slang import translate_slang, slang_coverage
from src.training.gold_loaders import (
    load_gold_sentiment, load_gold_toxicity, gold_toxicity_positives,
    oversample_minority, compute_class_weights, compute_pos_weights,
    SENTIMENT_LABELS, TOXICITY_LABELS,
)
from src.training.sentiment_trainer import fine_tune_sentiment
from src.training.toxicity_trainer import fine_tune_toxicity

In [4]:
# Sel 3: Models + hyperparameters + knobs
MODELS = {
    'bert': config['model_checkpoints']['bert']['hf_repo'],
    'roberta': config['model_checkpoints']['roberta']['hf_repo'],
    'distilbert': config['model_checkpoints']['distilbert']['hf_repo'],
}
HP_SENT = config['hyperparameters']['sentiment']
HP_TOX = config['hyperparameters']['toxicity']
MODELS_ROOT = Path('models')
MODELS_ROOT.mkdir(exist_ok=True)

# --- imbalance / preprocessing knobs (config-driven, with defaults) ---
prep_cfg = config.get('preprocessing', {})
APPLY_SLANG          = bool(prep_cfg.get('slang_translation', True))
OVERSAMPLE_FRAC      = float(prep_cfg.get('oversample_minority_frac', 0.30))
CLASS_WEIGHT_SCHEME  = str(prep_cfg.get('class_weight_scheme', 'balanced'))
POS_WEIGHT_CAP       = float(prep_cfg.get('pos_weight_cap', 50.0))
AUGMENT_TOX_WITH_GOLD= bool(prep_cfg.get('augment_toxicity_with_gold', True))

print('Models:')
for name, repo in MODELS.items():
    print(f'  {name:12s} -> {repo}')
print(f'\nslang_translation={APPLY_SLANG}  oversample_frac={OVERSAMPLE_FRAC}  '
      f'class_weight={CLASS_WEIGHT_SCHEME}  pos_weight_cap={POS_WEIGHT_CAP}  '
      f'augment_tox_with_gold={AUGMENT_TOX_WITH_GOLD}')

Models:
  bert         -> bert-base-uncased
  roberta      -> roberta-base
  distilbert   -> distilbert-base-uncased

slang_translation=True  oversample_frac=0.3  class_weight=balanced  pos_weight_cap=50.0  augment_tox_with_gold=True


## A. Sentiment — in-domain (gold), slang-aware, class-balanced

`load_gold_sentiment` reads `data/gold/train.csv`, applies `translate_slang`,
maps `sentiment -> {negative:0, neutral:1, positive:2}`, and splits train/val
(stratified). The held-out `data/gold/test.csv` is **not** touched here — it is
the gold-test used by notebook 06.


In [5]:
# Sel 4: Load in-domain sentiment + diagnostics
sent_split = load_gold_sentiment(gold_root='data/gold', apply_slang=APPLY_SLANG,
                                 val_frac=0.1, seed=SEED, label_names=SENTIMENT_LABELS)
print(f'train={len(sent_split.train):,}  val={len(sent_split.validation):,}  test={len(sent_split.test):,}')
print('labels:', sent_split.label_names)
print('train class dist:', sent_split.train["label"].value_counts().sort_index().to_dict(),
      '(0=neg, 1=neu, 2=pos)')

cov = slang_coverage(load_gold_sentiment(apply_slang=False).train["text"])
print(f'slang coverage on gold: msg_hit={cov["msg_hit_rate"]:.1%}  '
      f'token_hit={cov["token_hit_rate"]:.1%}  dict_size={int(cov["dict_size"])}')
print('\nexample (translated) inputs:')
for t, l in zip(sent_split.train["text"].head(6), sent_split.train["label"].head(6)):
    print(f'   [{l}] {t!r}')

train=5,021  val=558  test=2,392
labels: ['negative', 'neutral', 'positive']
train class dist: {0: 64, 1: 2115, 2: 2842} (0=neg, 1=neu, 2=pos)
slang coverage on gold: msg_hit=75.9%  token_hit=57.6%  dict_size=237

example (translated) inputs:
   [1] 'go engage attack now'
   [2] 'good game well played'
   [2] 'good game well played'
   [2] 'good game well played'
   [2] 'thank you'
   [2] 'akwkaokw'


In [6]:
# Sel 5: Oversample minority + class weights
# Strategy (researched): moderate oversampling of the rare 'negative' class up to
# OVERSAMPLE_FRAC x the majority count, THEN balanced weights on the resampled set
# for residual imbalance. Aggressive duplication is avoided (only ~64 unique
# negatives exist) — class weights pick up the slack.
import numpy as np

sent_train_bal = oversample_minority(sent_split.train, label_col='label',
                                     target_frac=OVERSAMPLE_FRAC, seed=SEED)
print('after oversample:', sent_train_bal["label"].value_counts().sort_index().to_dict(),
      '| total', len(sent_train_bal))

sent_class_weights = compute_class_weights(sent_train_bal["label"], n_classes=3,
                                           scheme=CLASS_WEIGHT_SCHEME)
print('class weights (neg, neu, pos):', np.round(sent_class_weights, 3).tolist())

# WARNING worth documenting in the thesis: only ~64 *distinct* negative messages
# exist. Oversampling + weights raise negative recall but cannot manufacture
# diversity — expect negative to remain the weakest class.
if (sent_split.train["label"] == 0).sum() < 100:
    msg = f'Only {(sent_split.train["label"]==0).sum()} distinct negative training messages — negative class is data-limited.'
    print(f'[WARN] {msg}'); run_log.add_warning(msg)

after oversample: {0: 853, 1: 2115, 2: 2842} | total 5810
class weights (neg, neu, pos): [2.27, 0.916, 0.681]
[WARN] Only 64 distinct negative training messages — negative class is data-limited.


In [7]:
# Sel 6: Fine-tune 3 models -> SENTIMENT (in-domain, weighted)
import json, time
SENT_F1_MACRO_THRESHOLD = 0.55  # in-domain target (3-class, imbalanced)
results = {}

for model_key, model_repo in MODELS.items():
    out_dir = MODELS_ROOT / f'{model_key}-sentiment'
    print(f'\n=== Fine-tune {model_key} -> SENTIMENT -> {out_dir} ===')
    t0 = time.time()
    res = fine_tune_sentiment(
        model_name=model_repo,
        train_df=sent_train_bal,
        val_df=sent_split.validation,
        label_names=sent_split.label_names,
        output_dir=out_dir,
        hyperparameters=HP_SENT,
        seed=SEED,
        class_weights=sent_class_weights.tolist(),
    )
    elapsed = time.time() - t0
    m = res.eval_metrics
    print(f'  done in {elapsed:.0f}s | bal_acc={m.get("eval_balanced_accuracy"):.3f} '
          f'mcc={m.get("eval_mcc"):.3f} f1_macro={m.get("eval_f1_macro"):.3f}')
    if m.get('eval_f1_macro', 0) < SENT_F1_MACRO_THRESHOLD:
        wmsg = f'{model_key}/sentiment val F1-macro={m.get("eval_f1_macro",0):.3f} < {SENT_F1_MACRO_THRESHOLD}'
        print(f'  [WARN] {wmsg}'); run_log.add_warning(wmsg)
    results[(model_key, 'sentiment')] = res
    log_path = Path('reports') / f'training_{model_key}_sentiment.log'
    log_path.write_text(json.dumps({'metrics': m, 'history': res.history,
                                    'duration_sec': elapsed}, indent=2, default=str))
    run_log.add_output(out_dir); run_log.add_output(log_path)


=== Fine-tune bert -> SENTIMENT -> models\bert-sentiment ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1369.96it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Mcc,F1 Macro,Precision Macro,Recall Macro
1,0.276982,0.176923,0.967742,0.838775,0.935942,0.838115,0.837528,0.838775
2,0.198100,0.183387,0.976703,0.798576,0.953465,0.852790,0.983267,0.798576
3,0.137148,0.219988,0.978495,0.799994,0.957119,0.854031,0.984367,0.799994


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

  done in 102s | bal_acc=0.800 mcc=0.957 f1_macro=0.854

=== Fine-tune roberta -> SENTIMENT -> models\roberta-sentiment ===


...

  done in 99s | bal_acc=0.846 mcc=0.957 f1_macro=0.876

=== Fine-tune distilbert -> SENTIMENT -> models\distilbert-sentiment ===


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1940.27it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 558/558 [00:00<00:00, 58567.64 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Mcc,F1 Macro,Precision Macro,Recall Macro
1,0.261980,0.205451,0.969534,0.840193,0.939447,0.853496,0.869338,0.840193
2,0.196802,0.176958,0.973118,0.749902,0.946347,0.799042,0.980602,0.749902
3,0.142764,0.186883,0.976703,0.798576,0.953465,0.852790,0.983267,0.798576


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.83s/it]

  done in 53s | bal_acc=0.799 mcc=0.953 f1_macro=0.853


## B. Toxicity — Jigsaw (external) + slang bridge + pos_weight

**Decision (documented):** the gold set cannot supervise a 6-label toxicity
classifier — `severe_toxic` and `threat` have **0** positives, `identity_hate`
only 2, and most gold-toxic messages are non-English profanity. So toxicity is
trained on **Jigsaw** (lots of positives), with two adaptations:

1. `translate_slang` applied to all text — so Dota toxic jargon (`noob`, `trash`,
   `ez`, `kys`) maps onto the toxic vocabulary Jigsaw was trained on.
2. Optionally augment with the handful of **English** gold toxic positives.

`pos_weight` (= n_neg / n_pos per label, capped) counters the heavy
non-toxic majority in BCE loss.


In [8]:
# Sel 7: Load Jigsaw + slang-translate + (optional) gold augmentation + pos_weights
from src.training.dataset_loaders import load_toxicity_dataset

try:
    tox_split = load_toxicity_dataset(source='jigsaw', cache_dir=Path('data/external'))
    print(f'Jigsaw: train={len(tox_split.train):,}  val={len(tox_split.validation):,}  test={len(tox_split.test):,}')
    print('labels:', tox_split.label_names)

    def _slang_df(df):
        df = df.copy()
        if APPLY_SLANG:
            df['text'] = df['text'].fillna('').astype(str).map(translate_slang)
        return df

    tox_train = _slang_df(tox_split.train)
    tox_val   = _slang_df(tox_split.validation)

    if AUGMENT_TOX_WITH_GOLD:
        gold_pos = gold_toxicity_positives('data/gold', apply_slang=APPLY_SLANG,
                                           label_cols=tox_split.label_names)
        # keep only rows that overlap Jigsaw label schema; ensure same columns
        gold_pos = gold_pos[['text'] + tox_split.label_names]
        tox_train = __import__('pandas').concat([tox_train[['text'] + tox_split.label_names], gold_pos],
                                                ignore_index=True)
        print(f'augmented with {len(gold_pos)} in-domain toxic positives -> train={len(tox_train):,}')

    tox_pos_weights = compute_pos_weights(tox_train, tox_split.label_names, cap=POS_WEIGHT_CAP)
    print('pos_weights:', dict(zip(tox_split.label_names, np.round(tox_pos_weights, 1).tolist())))
    print('label positives:', tox_train[tox_split.label_names].sum().astype(int).to_dict())
except FileNotFoundError as e:
    print(f'[ERROR] {e}')
    print('Download Jigsaw from Kaggle first, then re-run this cell.')
    tox_split = None

Jigsaw: train=143,614  val=15,957  test=63,978
labels: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
augmented with 54 in-domain toxic positives -> train=143,668
pos_weights: {'toxic': 9.399999618530273, 'severe_toxic': 50.0, 'obscene': 17.799999237060547, 'threat': 50.0, 'insult': 19.100000381469727, 'identity_hate': 50.0}
label positives: {'toxic': 13863, 'severe_toxic': 1447, 'obscene': 7652, 'threat': 441, 'insult': 7133, 'identity_hate': 1261}


In [9]:
# Sel 8: Fine-tune 3 models -> TOXICITY (Jigsaw + slang, pos_weighted)
TOX_F1_MICRO_THRESHOLD = 0.65
if tox_split is not None:
    for model_key, model_repo in MODELS.items():
        out_dir = MODELS_ROOT / f'{model_key}-toxicity'
        print(f'\n=== Fine-tune {model_key} -> TOXICITY -> {out_dir} ===')
        t0 = time.time()
        res = fine_tune_toxicity(
            model_name=model_repo,
            train_df=tox_train,
            val_df=tox_val,
            label_names=tox_split.label_names,
            output_dir=out_dir,
            hyperparameters=HP_TOX,
            seed=SEED,
            pos_weights=tox_pos_weights.tolist(),
        )
        elapsed = time.time() - t0
        m = res.eval_metrics
        print(f'  done in {elapsed:.0f}s | f1_micro={m.get("eval_f1_micro"):.3f} '
              f'pr_auc_macro={m.get("eval_pr_auc_macro"):.3f} mcc_macro={m.get("eval_mcc_macro"):.3f}')
        if m.get('eval_f1_micro', 0) < TOX_F1_MICRO_THRESHOLD:
            wmsg = f'{model_key}/toxicity val F1-micro={m.get("eval_f1_micro",0):.3f} < {TOX_F1_MICRO_THRESHOLD}'
            print(f'  [WARN] {wmsg}'); run_log.add_warning(wmsg)
        results[(model_key, 'toxicity')] = res
        log_path = Path('reports') / f'training_{model_key}_toxicity.log'
        log_path.write_text(json.dumps({'metrics': m, 'history': res.history,
                                        'duration_sec': elapsed}, indent=2, default=str))
        run_log.add_output(out_dir); run_log.add_output(log_path)
else:
    print('[SKIP] toxicity training — Jigsaw dataset unavailable.')


=== Fine-tune bert -> TOXICITY -> models\bert-toxicity ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 708.50it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tra

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Hamming Loss,Mcc Macro,Pr Auc Macro,Pr Auc Micro,Runtime,Samples Per Second,Steps Per Second
1,0.177225,0.180492,0.660936,0.543308,0.035429,0.586650,0.679472,0.832193,39.653600,402.410000,12.584000
2,0.189162,0.227707,0.750030,0.640538,0.022091,0.660552,0.707375,0.870910,38.608200,413.306000,12.925000
3,0.100593,0.236432,0.757035,0.655300,0.021192,0.670331,0.703536,0.862100,41.415200,385.293000,12.049000


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.21s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

  done in 4027s | f1_micro=0.757 pr_auc_macro=0.704 mcc_macro=0.670

=== Fine-tune roberta -> TOXICITY -> models\roberta-toxicity ===


...

  done in 4322s | f1_micro=0.750 pr_auc_macro=0.713 mcc_macro=0.659

=== Fine-tune distilbert -> TOXICITY -> models\distilbert-toxicity ===


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2108.94it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 15957/15957 [00:01<00:00, 13673.14 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Hamming Loss,Mcc Macro,Pr Auc Macro,Pr Auc Micro,Runtime,Samples Per Second,Steps Per Second
1,0.197496,0.189220,0.680700,0.559270,0.032003,0.597025,0.689575,0.857988,21.837000,730.733000,22.851000
2,0.219860,0.270169,0.763744,0.649517,0.020064,0.660580,0.698289,0.873970,21.959900,726.642000,22.723000
3,0.125692,0.248663,0.752933,0.641968,0.021777,0.657626,0.700533,0.873163,22.296200,715.683000,22.381000


Writing model shards: 100%|██████████| 1/1 [00:14<00:00, 14.91s/it]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.89s/it]

  done in 2308s | f1_micro=0.753 pr_auc_macro=0.701 mcc_macro=0.658


In [10]:
# Sel 9: Pin Hugging Face revision SHAs + write models/detoxify/manifest.json
import yaml
from huggingface_hub import HfApi
api = HfApi()

cfg_path = Path('configs/experiment.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
for model_key, repo in MODELS.items():
    try:
        info = api.model_info(repo)
        cfg['model_checkpoints'][model_key]['revision_sha'] = info.sha
        print(f'  {model_key}: {repo} @ {info.sha[:12]}')
    except Exception as e:
        wmsg = f'Failed to fetch revision SHA for {repo}: {e}'
        print(f'  [WARN] {wmsg}'); run_log.add_warning(wmsg)

detox_repo = cfg['model_checkpoints']['detoxify']['hf_repo']
detox_sha = ''
try:
    info = api.model_info(detox_repo); detox_sha = info.sha
    cfg['model_checkpoints']['detoxify']['revision_sha'] = info.sha
    print(f'  detoxify: @ {info.sha[:12]}')
except Exception as e:
    print(f'  [WARN] {e}')

detox_dir = MODELS_ROOT / 'detoxify'; detox_dir.mkdir(parents=True, exist_ok=True)
(detox_dir / 'manifest.json').write_text(json.dumps({
    'hf_repo': detox_repo, 'revision_sha': detox_sha,
    'note': 'pretrained, not fine-tuned — used directly in notebook 05 (toxicity specialist) and zero-shot sentiment mapping in notebook 06. Slang translation IS applied at inference for a fair comparison.',
    'task': 'toxicity_multi_label', 'labels': cfg['labels']['toxicity_labels'],
}, indent=2), encoding='utf-8')
print(f'  detoxify manifest -> {detox_dir / "manifest.json"}')
run_log.add_output(detox_dir / 'manifest.json')

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
run_log.add_output(cfg_path)

  bert: bert-base-uncased @ 86b5e0934494
  roberta: roberta-base @ e2da8e2f811d
  distilbert: distilbert-base-uncased @ 12040accade4
  detoxify: @ 4d6c22e74ba2
  detoxify manifest -> models\detoxify\manifest.json


In [11]:
run_log.save('reports/run_log.csv')

[run_log] 04_training → 10929.95s, 14 outputs, 1 warnings → reports\run_log.csv
